<h1 align="center">T2V · Text to Vision</h1>
<p align="center"><b>Describe a classifier in words. Get a working vision model: every weight generated, no training, no images.</b></p>
<p align="center">
<a href="https://github.com/kalidolinux-sudo/t2v-text-to-vision"><img src="https://img.shields.io/badge/GitHub-t2v--text--to--vision-181717?logo=github"></a>
<img src="https://img.shields.io/badge/agent-AMD%20TokenFactory-ED1C24">
<img src="https://img.shields.io/badge/model-T2V--1K%20(ImageNet--1k)-success">
<img src="https://img.shields.io/badge/license-MIT-blue">
</p>

An **agent on AMD TokenFactory** understands your request, describes every class visually and reasons about
look-alikes for classes it has never seen. A diffusion model (Tina × RPG, 49M parameters) then writes all
**274,042 weights of a ResNet-20** directly from the text.

| ImageNet-1k, 100 test tasks of 10 classes per row | **T2V-1K (from text)** | Fine-tuned on images | Share |
|---|---|---|---|
| Seen classes, new combinations | **70.0%** | 72.4% | 97% |
| All classes never seen | **63.6%** | 74.1% | 86% |
| Never seen, whole groups held out | **47.5%** | 64.8% | 73% |

### How to use
`Runtime → Run all`. A GPU is optional (`Runtime → Change runtime type → T4 GPU` makes it much faster).
Change the classes in **step 1** and run the cells below it again.

### Contents
0. Setup · load T2V-1K
1. Your request → class names
2. Agent: visual descriptions
3. Known or new? The known-class gate
4. Agent: analogies for new classes
5. Generate all the weights
6. Test on real ImageNet photos
7. Does the text really drive the weights?
8. What each part adds (on your task)
9. Try your own photo
10. Export the model
11. Benchmark results of T2V-1K
12. The full app on a public link

## 0 · Setup
Clones the repository, installs three small packages and downloads the T2V-1K generator (198 MB) from the
GitHub release. Nothing here needs an account.

In [ ]:
#@title **0.1 · Install** (a few minutes the first time)
import os, subprocess, sys, warnings
os.environ.update(TRANSFORMERS_VERBOSITY="error", HF_HUB_DISABLE_PROGRESS_BARS="1", TOKENIZERS_PARALLELISM="false")
warnings.filterwarnings("ignore")
REPO = "https://github.com/kalidolinux-sudo/t2v-text-to-vision"
if not os.path.isdir("/content/t2v-text-to-vision"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/content/t2v-text-to-vision"], check=True)
os.chdir("/content/t2v-text-to-vision")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore", "gradio>=4.0", "onnx", "pyarrow"],
               check=True)

import torch, transformers, huggingface_hub
transformers.logging.set_verbosity_error(); transformers.logging.disable_progress_bar()
huggingface_hub.utils.logging.set_verbosity_error()
from app import RELEASE_URL, download
CKPT = "checkpoints/t2v_1k.pt"
if not os.path.exists(CKPT):
    download(RELEASE_URL, CKPT)          # T2V-1K generator from the GitHub release
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU (works; a T4 GPU is much faster)")

In [ ]:
#@title **0.2 · Agent key** (optional)
#@markdown The agent runs on **AMD TokenFactory** (DeepSeek-V4-Flash). Add your key as a Colab secret named
#@markdown `TOKENFACTORY_API_KEY` (🔑 icon on the left), or tick the box to paste it.
#@markdown Without a key everything still works: the request is split on commas, and the TokenFactory
#@markdown descriptions and analogies stored with the model are used.
ask_for_key = False  #@param {type:"boolean"}
from getpass import getpass
key = None
try:
    from google.colab import userdata
    key = userdata.get("TOKENFACTORY_API_KEY")
except Exception:
    pass
if not key and ask_for_key:
    key = getpass("AMD TokenFactory API key (press Enter to skip): ").strip()
if key:
    os.environ["TOKENFACTORY_API_KEY"] = key
os.environ.setdefault("TOKENFACTORY_BASE_URL", "https://developer.amd.com.cn/radeon/api/v1")
os.environ.setdefault("TOKENFACTORY_MODEL", "DeepSeek-V4-Flash")
del key
LIVE = bool(os.environ.get("TOKENFACTORY_API_KEY"))
print("agent: AMD TokenFactory (live)" if LIVE else "agent: offline (TokenFactory outputs stored with the model)")

In [ ]:
#@title **0.3 · Load T2V-1K** (generator + CLIP text encoder)
import json, time, random, numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from t2v import target
from t2v.generate import load_generator, generate_flat, build_model, export
from t2v.llm import parse_classes
from t2v.describe import describe
from t2v.text_embed import embed_names
from t2v.data import _load_imagenet

gen, tok, ck = load_generator(CKPT, DEV)
NAMES = ck["names"]
CANON = {n.lower(): n for n in NAMES}
SEEN = [NAMES[i] for i in gen.seen_ids]
NEW_IN_VOCAB = sorted(set(NAMES) - set(SEEN))
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def make(classes, **kw):
    # generate one model for `classes`; returns (model, info, seconds, flat weights)
    info, t0 = {}, time.time()
    flat = generate_flat(gen, tok, classes, steps=5, body_noise=0.0, info=info, amp=DEV == "cuda",
                         **{"gate": "auto", **kw})[0]
    if DEV == "cuda":
        torch.cuda.synchronize()
    return build_model(ck["arch"], flat, len(classes)).to(DEV), info, time.time() - t0, flat

_VAL = {}
def photos(classes):
    # real ImageNet validation photos (32x32, 50 per class, never used in training) of these classes
    if not _VAL:
        _VAL.update(_load_imagenet("test", "data", 260))
    ids = [NAMES.index(c) if c in NAMES else None for c in classes]
    if None in ids:
        return None
    m = torch.isin(_VAL["y"], torch.tensor(ids))
    return _VAL["x"][m], torch.tensor([ids.index(int(v)) for v in _VAL["y"][m]])

@torch.no_grad()
def predict(model, x):
    return torch.cat([model(x[i:i + 500].float().div(255).to(DEV)).argmax(1).cpu() for i in range(0, len(x), 500)])

n_gen = sum(p.numel() for p in gen.parameters())
display(Markdown("\n".join([
    "| T2V-1K | |", "|---|---|",
    f"| generator | {n_gen/1e6:.1f}M parameters · LSTM over parameter tokens + joint transformer + MLP denoiser · 5 DDIM steps |",
    f"| generates | `{ck['arch']}` · {tok.num_params:,} weights · up to {tok.K} classes |",
    f"| vocabulary | {len(NAMES)} ImageNet classes · {len(SEEN)} seen in training · {len(NEW_IN_VOCAB)} never seen |",
    f"| known-class gate | CLIP similarity ≥ {gen.gate} to a training class |",
    f"| TokenFactory analogies | stored for {len(gen.neighbors['store'])} classes · blend β = {gen.neighbors['llm_beta']} (cross-validated) |",
])))

## 1 · Your request → class names
Type the classes your vision model should tell apart (2 to 10). With a TokenFactory key you can write a
free-form request in any language, e.g. `أبي نموذج يميز بين الفيل والبيانو والحوت القاتل`.

The default mixes **three classes the generator never saw in training** (African elephant, grand piano,
killer whale) with five it knows. It is one example task; the averages over many tasks are in section 11.

In [ ]:
#@title **1 · Request**
request = "African elephant, grand piano, killer whale, pizza, tabby, school bus, strawberry, golden retriever"  #@param {type:"string"}
classes, pinfo = parse_classes(request, tok.K)
classes = [CANON.get(c.strip().lower(), c) for c in classes]       # match the vocabulary ("African elephant")
assert len(classes) >= 2, "need at least 2 classes"
src = "offline (split on commas)" if pinfo.get("source", "").startswith("fallback") else f"AMD TokenFactory · {pinfo['source']}"
display(Markdown(f"**Parsed by:** {src}" + (f" · task: *{pinfo['task']}*" if pinfo.get("task") else "") + "\n\n"
                 + "\n".join(f"{i+1}. **{c}**" + ("" if c in NAMES else " *(not an ImageNet class name: generated from text, no photo test)*")
                             for i, c in enumerate(classes))))

## 2 · Agent: visual descriptions
A bare class name gives CLIP little to work with. The agent writes five short visual descriptions per class
(CuPL); the generator is conditioned on the average CLIP embedding of the name and its descriptions.
TokenFactory described all 1,000 ImageNet classes once (stored with the model); new names are described live.

In [ ]:
#@title **2 · Descriptions**
parsed = pinfo.get("descriptions") or {}
descs, origin = {}, {}
for c in classes:
    d = parsed.get(c) or parsed.get(c.lower())
    if d:
        descs[c], origin[c] = d, "TokenFactory (live, with the request)"
    elif (gen.descriptions or {}).get(c):
        descs[c], origin[c] = gen.descriptions[c], "TokenFactory (stored)"
missing = [c for c in classes if c not in descs]
if missing and LIVE:
    for c, d in describe(missing).items():
        descs[c], origin[c] = d, "TokenFactory (live)"
rows = ["| class | written by | descriptions |", "|---|---|---|"]
for c in classes:
    shown = "<br>".join(f"• {d}" for d in (descs.get(c) or [])[:3]) or "— (class name only)"
    rows.append(f"| **{c}** | {origin.get(c, '—')} | {shown} |")
display(Markdown("\n".join(rows)))

## 3 · Known or new? The known-class gate
For a class it was trained on, the generator writes a precise classifier row. For a class it never saw, it
tends to write the remembered row of some similar known class, which hurts. So each class is checked first:
if its text embedding is (almost) identical to a training class, it counts as **known**; otherwise it is
**new** and gets a row built from its text and its visual analogies (step 4). The threshold was set
automatically during training.

In [ ]:
#@title **3 · Gate**
with torch.no_grad():
    emb = embed_names(classes, DEV, ensemble=gen.prompt_ensemble, descriptions=descs)
    best, arg = (emb @ gen.seen_emb.T).max(1)
best, arg = best.cpu(), arg.cpu()
known = (best >= gen.gate).tolist()
rows = ["| class | closest training class | CLIP similarity | verdict |", "|---|---|---|---|"]
for c, s, a, k in zip(classes, best.tolist(), arg.tolist(), known):
    rows.append(f"| {c} | {SEEN[a]} | {s:.4f} | {'known' if k else '**new: never seen in training**'} |")
display(Markdown("\n".join(rows)))

fig, ax = plt.subplots(figsize=(8, 0.42 * len(classes) + 1))
ax.barh(classes[::-1], best.tolist()[::-1], color=["#059669" if k else "#e11d48" for k in known[::-1]])
ax.axvline(gen.gate, color="black", ls="--", lw=1); ax.text(gen.gate, len(classes) - 0.4, f" gate {gen.gate}", fontsize=8)
ax.set_xlim(max(0.0, best.min().item() - 0.05), 1.002); ax.set_xlabel("CLIP similarity to the closest training class")
ax.set_title("green = known · red = new"); plt.tight_layout(); plt.show()

## 4 · Agent: analogies for new classes
For each new class the agent picks the known classes it **looks like**, with weights. Their learned
classifier rows are mixed into the new class's row (β chosen by cross-validation on 100 validation classes).
CLIP proposes 20 candidates; TokenFactory reasons about which ones really look alike. Analogies for the 100
held-out ImageNet classes are stored with the model; other new names are asked live.

In [ ]:
#@title **4 · Analogies**
from t2v import neighbors as N
store = gen.neighbors["store"]
new = [c for c, k in zip(classes, known) if not k]
asked = [c for c in new if c not in store]
if asked and LIVE:
    idx = [classes.index(c) for c in asked]
    cands = N.shortlist(emb[idx], gen.seen_emb, SEEN, 20)
    store.update(N.ask(asked, dict(zip(asked, cands))))
if not new:
    display(Markdown("All classes are known: no analogies needed. Try a class from the list below."))
else:
    rows = [f"| new class | looks like (TokenFactory) | source |", "|---|---|---|"]
    for c in new:
        a = store.get(c)
        look = ", ".join(f"**{n}** {w:.0%}" for n, w in a.items()) if a else "— (text prior only; add a key for live analogies)"
        rows.append(f"| {c} | {look} | {'live' if c in asked and a else 'stored' if a else '—'} |")
    display(Markdown("\n".join(rows) + f"\n\nRow of a new class = (1 − β) · text prior + β · mix of these rows, β = {gen.neighbors['llm_beta']}"))
display(Markdown("<details><summary>ImageNet classes never seen in training (try them in step 1)</summary>\n\n"
                 + ", ".join(NEW_IN_VOCAB) + "\n</details>"))

## 5 · Generate all the weights
The diffusion model denoises the whole network as a sequence of parameter tokens, conditioned on one CLIP
token per class: convolution filters, batch-norm statistics and the classifier, 274,042 numbers in total.
No gradient step and no image is involved.

In [ ]:
#@title **5 · Generate**
model, info, dt, flat = make(classes, descriptions=descs)
classes = info.get("names", classes)
groups = {}
for name, shape in target.param_spec(model.net):
    p = name.split(".")
    g = ("stem" if p[0] in ("conv1", "bn1") else f"stage {int(p[1]) // 3 + 1}" if p[0] == "layers" else "classifier")
    groups[g] = groups.get(g, 0) + int(torch.Size(shape).numel())
display(Markdown(f"### Generated a `{ck['arch']}` with **{flat.numel():,} weights** in **{dt:.2f} s** on `{DEV}`\n\n"
                 "| part | weights |\n|---|---|\n" + "\n".join(f"| {g} | {n:,} |" for g, n in groups.items())))
w = flat.cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(12, 2.8))
ax[0].hist(np.clip(w, -1, 1), bins=200, color="#2563eb"); ax[0].set_yscale("log"); ax[0].set_title("all generated weights (clipped to ±1)")
rows_ = target.head_rows(ck["arch"], flat[None].cpu())[0, :len(classes), :-1]
im = ax[1].imshow(rows_.numpy(), aspect="auto", cmap="coolwarm", vmin=-rows_.abs().max(), vmax=rows_.abs().max())
ax[1].set_yticks(range(len(classes))); ax[1].set_yticklabels(classes, fontsize=8); ax[1].set_title("generated classifier rows (one per class)")
plt.colorbar(im, ax=ax[1]); plt.tight_layout(); plt.show()

## 6 · Test on real ImageNet photos
The generated model now classifies the 50 ImageNet validation photos of each class (32×32, never used in
training). The validation set is downloaded once.

In [ ]:
#@title **6.1 · Accuracy**
data = photos(classes)
if data is None:
    display(Markdown("*Some classes are not ImageNet-1k names, so there are no photos to test on. The model was still generated (step 10 exports it).*"))
else:
    x, y = data
    pred = predict(model, x)
    acc = (pred == y).float().mean().item()
    per = [(pred[y == j] == j).float().mean().item() for j in range(len(classes))]
    display(Markdown(f"### Accuracy on **{len(y)} real ImageNet photos**: **{acc:.1%}** (chance {1/len(classes):.0%})"))
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(classes) + 1))
    ax.barh(classes[::-1], per[::-1], color=["#059669" if k else "#e11d48" for k in known[::-1]])
    ax.axvline(1 / len(classes), color="gray", ls=":", lw=1); ax.set_xlim(0, 1)
    for i, a in enumerate(per[::-1]):
        ax.text(a + 0.01, i, f"{a:.0%}", va="center", fontsize=8)
    ax.set_title("accuracy per class · green = known · red = never seen in training"); plt.tight_layout(); plt.show()

In [ ]:
#@title **6.2 · Predictions** (green = correct)
if data is not None:
    pick = torch.randperm(len(y), generator=torch.Generator().manual_seed(0))[:16]
    fig, axes = plt.subplots(2, 8, figsize=(16, 5.6))
    for ax, i in zip(axes.flat, pick.tolist()):
        ax.imshow(x[i].permute(1, 2, 0).numpy(), interpolation="nearest")
        ok = pred[i] == y[i]
        ax.set_title(f"✓ {classes[pred[i]]}" if ok else f"✗ {classes[pred[i]]}\n(is {classes[y[i]]})",
                     color="green" if ok else "red", fontsize=9)
        ax.axis("off")
    fig.suptitle("Predictions of the generated model · 32×32 ImageNet validation photos", fontsize=11)
    plt.tight_layout(h_pad=1.5); plt.show()

In [ ]:
#@title **6.3 · Confusion matrix**
if data is not None:
    k = len(classes)
    cm = torch.zeros(k, k)
    for t, p in zip(y.tolist(), pred.tolist()):
        cm[t, p] += 1
    cm = cm / cm.sum(1, keepdim=True)
    fig, ax = plt.subplots(figsize=(0.75 * k + 2, 0.65 * k + 1.5))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(k):
        for j in range(k):
            if cm[i, j] >= 0.05:
                ax.text(j, i, f"{cm[i, j]:.0%}", ha="center", va="center", fontsize=7, color="white" if cm[i, j] > 0.5 else "black")
    ax.set_xticks(range(k)); ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(k)); ax.set_yticklabels(classes, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("where the generated model is confused")
    plt.tight_layout(); plt.show()

## 7 · Does the text really drive the weights?
If the generator ignored the text, any class names would give the same accuracy. Here a second model is
generated for **wrong names** (random training classes) and tested on the same photos; it should fall to
chance. A third model gets the **same names in reversed order**; its outputs should simply follow the order.

In [ ]:
#@title **7 · Sanity checks**
if data is not None:
    pool = [c for c in SEEN if c not in classes]
    wrong = random.Random(0).sample(pool, len(classes))
    m_wrong = make(wrong)[0]
    acc_wrong = (predict(m_wrong, x) == y).float().mean().item()
    m_rev = make(classes[::-1], descriptions=descs)[0]
    acc_rev = (predict(m_rev, x) == (len(classes) - 1 - y)).float().mean().item()
    res = {"right names": acc, "right names, reversed order": acc_rev, "wrong names": acc_wrong, "chance": 1 / len(classes)}
    fig, ax = plt.subplots(figsize=(8, 2.4))
    ax.barh(list(res)[::-1], list(res.values())[::-1], color=["gray", "#e11d48", "#2563eb", "#059669"])
    for i, v in enumerate(list(res.values())[::-1]):
        ax.text(v + 0.01, i, f"{v:.1%}", va="center", fontsize=9)
    ax.set_xlim(0, 1); ax.set_title("same photos, models generated from different text"); plt.tight_layout(); plt.show()
    display(Markdown(f"Wrong names used: *{', '.join(wrong)}*"))

## 8 · What each part adds (on your task)
The same request is generated with parts switched off. On a single task the differences are noisy; the
averages over 100 tasks per split are in section 11 and in the repository README.

In [ ]:
#@title **8 · Ablation**
if data is not None:
    variants = {
        "T2V: TokenFactory analogies (default)": dict(prior="llm"),
        "CLIP neighbours instead of the agent": dict(prior="clip"),
        "text prior only (no analogies)": dict(prior="ridge"),
        "no known-class gate": dict(gate=None),
    }
    is_new = torch.tensor([not known[int(t)] for t in y])
    out = {}
    for name, kw in variants.items():
        p = predict(make(classes, descriptions=descs, **kw)[0], x)
        out[name] = ((p == y).float().mean().item(), (p[is_new] == y[is_new]).float().mean().item() if is_new.any() else float("nan"))
    rows = ["| variant | all photos | photos of new classes |", "|---|---|---|"]
    rows += [f"| {n} | {a:.1%} | {b:.1%} |" if b == b else f"| {n} | {a:.1%} | — |" for n, (a, b) in out.items()]
    display(Markdown("\n".join(rows)))

## 9 · Try your own photo
Upload a photo or paste an image URL. The model works at 32×32 (ImageNet32), so a clear, centred object
works best. Leave both empty to use a random validation photo.

In [ ]:
#@title **9 · Your photo**
image_url = ""  #@param {type:"string"}
upload = False  #@param {type:"boolean"}
import io, requests
from PIL import Image
img = None
if upload:
    from google.colab import files
    up = files.upload()
    img = Image.open(io.BytesIO(next(iter(up.values()))))
elif image_url:
    img = Image.open(io.BytesIO(requests.get(image_url, timeout=30, headers={"User-Agent": "Mozilla/5.0"}).content))
elif data is not None:
    i = int(torch.randint(len(y), (1,)))
    img = Image.fromarray(x[i].permute(1, 2, 0).numpy())
if img is not None:
    arr = np.asarray(img.convert("RGB"))
    small = F.interpolate(torch.from_numpy(arr).permute(2, 0, 1)[None].float() / 255, size=(32, 32),
                          mode="bicubic", align_corners=False).clamp(0, 1)
    with torch.no_grad():
        prob = model(small.to(DEV)).softmax(-1)[0].cpu()
    order = prob.argsort(descending=True)[:5].tolist()
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.2), gridspec_kw={"width_ratios": [1, 1, 1.6]})
    ax[0].imshow(arr); ax[0].set_title("your photo"); ax[0].axis("off")
    ax[1].imshow(small[0].permute(1, 2, 0).numpy(), interpolation="nearest"); ax[1].set_title("what the model sees (32×32)"); ax[1].axis("off")
    ax[2].barh([classes[j] for j in order][::-1], [prob[j].item() for j in order][::-1], color="#2563eb")
    ax[2].set_xlim(0, 1); ax[2].set_title(f"prediction: {classes[order[0]]} ({prob[order[0]]:.0%})")
    plt.tight_layout(); plt.show()

## 10 · Export the model
The generated network is an ordinary PyTorch model: `weights.pt` (state dict), `classes.json` and
`model.onnx` (input: images in [0, 1], 3×32×32; normalisation is built in).

In [ ]:
#@title **10 · Export**
import shutil
model.cpu()
onnx_path = export(model, classes, ck["arch"], "t2v_model")
model.to(DEV)
for f in sorted(os.listdir("t2v_model")):
    print(f"t2v_model/{f:<14} {os.path.getsize(os.path.join('t2v_model', f)) / 1e6:6.2f} MB")
zip_path = shutil.make_archive("t2v_model", "zip", "t2v_model")
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("saved", zip_path)

## 11 · Benchmark results of T2V-1K
From the training run on one Colab T4 (72 minutes): 1,000 fine-tuned ResNet-20s on ImageNet-1k tasks,
900 classes seen and 100 never seen. Each split is 100 test tasks of 10 classes; the oracle is a ResNet-20
fine-tuned on the images of each task. Full log: `results/t2v_1k/`, experiment history: `docs/EXPERIMENTS.md`.

In [ ]:
#@title **11 · Results**
splits = ["seen classes,\nnew combinations", "half the classes\nnever seen", "all classes\nnever seen", "never seen,\nwhole groups held out"]
t2v_acc, oracle = [70.0, 64.1, 63.6, 47.5], [72.4, 74.9, 74.1, 64.8]
d = json.load(open("results/t2v_1k/diagnose_t2v_1k.json")) if os.path.exists("results/t2v_1k/diagnose_t2v_1k.json") else None
fig, ax = plt.subplots(1, 2 if d else 1, figsize=(14 if d else 8, 3.6), squeeze=False)
a = ax[0, 0]; xs = np.arange(len(splits))
a.bar(xs - 0.2, oracle, 0.4, label="fine-tuned on images (oracle)", color="#94a3b8")
a.bar(xs + 0.2, t2v_acc, 0.4, label="T2V-1K, generated from text", color="#e11d48")
for i in range(len(splits)):
    a.text(xs[i] + 0.2, t2v_acc[i] + 1, f"{t2v_acc[i]:.1f}", ha="center", fontsize=8)
a.axhline(10, color="gray", ls=":", lw=1); a.set_xticks(xs); a.set_xticklabels(splits, fontsize=8)
a.set_ylabel("accuracy, % (10-way)"); a.set_ylim(0, 85); a.legend(fontsize=8); a.set_title("T2V-1K vs. fine-tuning")
if d:
    b = ax[0, 1]
    keys, lab = ["eval_seen", "eval_novel", "eval_novel_hard"], ["seen", "never seen", "never seen, hard"]
    right = [100 * d[k]["1_conditioning"]["right_classes"] for k in keys]
    wrong = [100 * d[k]["1_conditioning"]["wrong_classes"] for k in keys]
    xs = np.arange(3)
    b.bar(xs - 0.2, right, 0.4, label="right class names", color="#059669")
    b.bar(xs + 0.2, wrong, 0.4, label="wrong class names", color="#94a3b8")
    b.axhline(10, color="gray", ls=":", lw=1); b.set_xticks(xs); b.set_xticklabels(lab, fontsize=8); b.set_ylim(0, 85)
    b.legend(fontsize=8); b.set_title("diagnostics: the text drives the weights (50 tasks per split)")
plt.tight_layout(); plt.show()
display(Markdown('''
| Prior for classifier rows of new classes | Half new | All new | All new, hard |
|---|---|---|---|
| Ridge map from class text only | 64.0% | 61.2% | **48.3%** |
| + CLIP nearest neighbours (no LLM) | 64.2% | 62.9% | 48.1% |
| **+ TokenFactory visual analogies (T2V)** | 64.1% | **63.6%** | 47.5% |
| No known-class gate | 64.3% | 60.2% | 44.1% |
'''))

## 12 · The full app on a public link
The Gradio app: request → agent → generated model → classify any photo, download `weights.pt` / ONNX.
Click the `https://….gradio.live` link printed below (valid while this cell runs; stop the cell to shut it down).

In [ ]:
#@title **12 · Open the app**
subprocess.run([sys.executable, "app.py", "--ckpt", CKPT, "--share"])

---
### Links
- **Code, results, experiment log:** https://github.com/kalidolinux-sudo/t2v-text-to-vision
- **Model (T2V-1K):** https://github.com/kalidolinux-sudo/t2v-text-to-vision/releases/tag/1.0
- **Reproduce the training (72 min on a T4):** `notebooks/t2v_colab_final_imagenet.py`

**How this was built.** By its author with AI assistants: **Gemini** (Google's latest thinking model) helped
formulate the experiments and assisted with the research; **Claude** (Anthropic) wrote much of the code under
the author's direction; the author directed the project, ran the training on Google Colab and reviewed the
code, experiments and claims. **AMD TokenFactory** (DeepSeek-V4-Flash) is part of the system itself.

Built for the AMD Developer Hackathon: ACT III · MIT license